In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# =============================
# Config & paths
# =============================
SEED = 42
rng = np.random.default_rng(SEED)

base_sim   = "../../../data/simulation/"
train_dir  = "../../../data/training/"
test_dir   = "../../../data/test/"
final_dir  = "../../../data/final test/"
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)
os.makedirs(final_dir, exist_ok=True)

# Simulation sources (X: (N,30,4) or (N,4,4); y: strings, need mapping)
sim_paths = {
    "start":    (os.path.join(base_sim, "engine_start_X.npy"),    os.path.join(base_sim, "engine_start_y.npy")),
    "off":      (os.path.join(base_sim, "engine_off_X.npy"),      os.path.join(base_sim, "engine_off_y.npy")),
    "occ":      (os.path.join(base_sim, "engine_occ_train_X.npy"),      os.path.join(base_sim, "engine_occ_train_y.npy")),
    "normal":   (os.path.join(base_sim, "engine_normal_load_X.npy"),   os.path.join(base_sim, "engine_normal_load_y.npy")),
    "high":     (os.path.join(base_sim, "engine_high_load_X.npy"),     os.path.join(base_sim, "engine_high_load_y.npy")),
    "critical": (os.path.join(base_sim, "engine_critical_load_X.npy"), os.path.join(base_sim, "engine_critical_load_y.npy")),
}

# =============================
# Direct label map (strings -> ints) for SIM ONLY
# No fallback. If a string is not here, we raise.
# =============================
LABEL_MAP = {
    # Engine Start / Off
    "Engine Start": 0,
    "Engine Off (cold)": 1,
    "Engine Off (cooling)": 1,

    # Normal Load
    "NormalLoad (idle)": 2,
    "NormalLoad (accelerating)": 2,
    "NormalLoad (decelerating)": 2,
    "NormalLoad": 2,   # if appears without subtype

    # High Load
    "HighLoad (idle)": 2,
    "HighLoad (accelerating)": 2,
    "HighLoad (decelerating)": 2,
    "HighLoad": 2,
    "High": 2,          # if appears as just "High"

    # Critical Load
    "CriticalLoad (idle)": 2,
    "CriticalLoad (accelerating)": 2,
    "CriticalLoad (decelerating)": 2,
    "CriticalLoad": 2,

    # Unknown
    "Unknown": 3,
}

# =============================
# Helpers
# =============================
def ensure_numpy(x):
    return np.asarray(x, dtype=np.float32)

def normalize_to_4x4(X):
    """
    Accepts X that may be:
      - (N,30,4) -> split each into 7 windows of shape (4,4)
      - (N,4,4)  -> keep as-is
      - object/ragged of the above
    Returns dense array (M,4,4).
    """
    X = np.asarray(X, dtype=object)
    out = []
    for s in X:
        a = ensure_numpy(s)
        if a.shape == (30, 4):
            for start in range(0, 30, 4):
                if start + 4 <= 30:
                    out.append(a[start:start+4, :])  # (4,4)
        elif a.shape == (4, 4):
            out.append(a)
        else:
            raise ValueError(f"Unexpected sample shape {a.shape}; expected (30,4) or (4,4)")
    return np.asarray(out, dtype=np.float32)

def _to_text(v):
    if isinstance(v, (bytes, bytearray)):
        try:
            v = v.decode("utf-8", errors="ignore")
        except Exception:
            pass
    if isinstance(v, np.ndarray):
        v = v.reshape(-1)[0] if v.size else ""
    if isinstance(v, np.generic):
        v = v.item()
    return str(v)

def map_label_to_int_sim_only(v):
    """
    Mapping for SIMULATED labels:
    - If v is numeric (int-like), pass through (no remap).
    - If v is string, map via LABEL_MAP strictly.
    - Otherwise raise an error. No default to 3.
    """
    # Numeric passthrough
    if isinstance(v, (int, np.integer)):
        iv = int(v)
        if iv not in (0,1,2,3):
            raise ValueError(f"Numeric label out of range: {iv}")
        return iv

    # String mapping
    txt = _to_text(v)
    if txt in LABEL_MAP:
        return LABEL_MAP[txt]

    # Strict: do not silently map to Unknown
    raise ValueError(f"Unmapped simulated label: {txt!r}. Add it to LABEL_MAP.")

def load_sim_group(x_path, y_path):
    if not (os.path.exists(x_path) and os.path.exists(y_path)):
        return None, None
    X = np.load(x_path, allow_pickle=True)  # (N,30,4) or (N,4,4) or object
    y = np.load(y_path, allow_pickle=True)  # strings or possibly arrays of strings
    return X, y

def labels_from_y_for_4x4_windows_sim(X_raw, Y_raw):
    """
    For SIMULATED data only.
    For each original sequence in X_raw:
      - If shape (30,4) -> produces 7 windows (4,4); broadcast one label to the 7 windows
      - If shape (4,4)  -> produces 1 window; broadcast one label
    Label choice:
      - If y is scalar/string/number -> map directly (numeric passthrough)
      - If y is an array -> take the CENTER element (more stable than first)
    Strict: no fallback to 3; raise if unmapped string appears.
    """
    labs = []
    X_raw = np.asarray(X_raw, dtype=object)
    Y_raw = np.asarray(Y_raw, dtype=object)
    for i, s in enumerate(X_raw):
        a = ensure_numpy(s)
        # pick y item; if lengths mismatch, use last available
        y_item = Y_raw[i] if i < len(Y_raw) else (Y_raw[-1] if len(Y_raw) > 0 else "Unknown")
        # reduce to scalar if per-timestep
        if isinstance(y_item, np.ndarray) and y_item.size > 1:
            flat = y_item.reshape(-1)
            y_scalar = flat[len(flat)//2]  # center timestep
        else:
            y_scalar = y_item

        lab = map_label_to_int_sim_only(y_scalar)

        # number of windows generated by this sequence
        if a.shape == (30, 4):
            n = 7
        elif a.shape == (4, 4):
            n = 1
        else:
            raise ValueError(f"Unexpected sample shape {a.shape}; expected (30,4) or (4,4)")

        labs.extend([lab] * n)
    return np.asarray(labs, dtype=np.int64)

def concat_many(arrs):
    arrs = [a for a in arrs if a is not None and len(a) > 0]
    return np.concatenate(arrs, axis=0) if arrs else None

def stratified_split(X, y, seed):
    """
    70/15/15 split. Uses stratify if possible, otherwise falls back to random split.
    """
    _, class_counts = np.unique(y, return_counts=True)
    can_strat = np.all(class_counts >= 2) and (len(np.unique(y)) > 1)
    if can_strat:
        X_train, X_tmp, y_train, y_tmp = train_test_split(
            X, y, test_size=0.30, random_state=seed, stratify=y
        )
        # second split also stratified (may still fail in edge cases)
        X_test, X_final, y_test, y_final = train_test_split(
            X_tmp, y_tmp, test_size=0.50, random_state=seed, stratify=y_tmp
        )
    else:
        X_train, X_tmp, y_train, y_tmp = train_test_split(
            X, y, test_size=0.30, random_state=seed, shuffle=True
        )
        X_test, X_final, y_test, y_final = train_test_split(
            X_tmp, y_tmp, test_size=0.50, random_state=seed, shuffle=True
        )
        print("[WARN] Not enough samples per class to stratify. Performed random split.")
    return (X_train, y_train), (X_test, y_test), (X_final, y_final)

# =============================
# Load EXISTING Router splits (already numeric labels: 0/1/2/3) — DO NOT REMAP
# =============================
def load_router_split(x_path, y_path):
    if not (os.path.exists(x_path) and os.path.exists(y_path)):
        return None, None
    X = np.load(x_path, allow_pickle=True)
    y = np.load(y_path, allow_pickle=True)
    if X.dtype == object:
        X = np.stack([np.asarray(s, np.float32).reshape(4,4) for s in X], 0)
    else:
        X = X.astype(np.float32)
    # numeric labels passthrough
    if y.dtype == object:
        y = np.array([int(np.asarray(v).reshape(-1)[0]) for v in y], dtype=np.int64)
    else:
        y = y.astype(np.int64).reshape(-1)
    # sanity
    if not np.all(np.isin(y, [0,1,2,3])):
        bad = np.unique(y[~np.isin(y, [0,1,2,3])])
        raise ValueError(f"Router split contains invalid numeric labels: {bad}")
    return X, y

def combine(aX, ay, bX, by):
    if aX is None:  return bX, by
    if bX is None:  return aX, ay
    Xc = np.concatenate([aX, bX], axis=0).astype(np.float32)
    yc = np.concatenate([ay.reshape(-1), by.reshape(-1)], axis=0).astype(np.int64)
    return Xc, yc

# =============================
# 1) Load simulated groups
# =============================
sim_loaded = {}
for key, (xp, yp) in sim_paths.items():
    Xg, Yg = load_sim_group(xp, yp)
    sim_loaded[key] = (Xg, Yg)

if all(v[0] is None for v in sim_loaded.values()):
    raise FileNotFoundError("No simulation .npy files found under ../../../data/simulation/")

# =============================
# 2) Normalize SIM X to (4,4) and map SIM y strings/ints -> ints per (4,4) window
#     (Strict mapping; numeric passthrough; no default-to-3)
# =============================
parts_X, parts_y = [], []

def handle_group(name):
    Xg, Yg = sim_loaded[name]
    if Xg is None or Yg is None:
        return
    X4 = normalize_to_4x4(Xg)
    y4 = labels_from_y_for_4x4_windows_sim(Xg, Yg)  # strict
    # sanity: OCC should be 3 if its labels are "Unknown" or 3 upstream
    if name == "occ":
        uniq = np.unique(y4)
        if not np.all(uniq == 3):
            print(f"[WARN] OCC produced non-3 labels: {uniq.tolist()} (check your occ y)")
    parts_X.append(X4); parts_y.append(y4)

for key in ["start", "off", "normal", "high", "critical", "occ"]:
    handle_group(key)

X_sim = concat_many(parts_X)
y_sim = concat_many(parts_y)
if X_sim is None or y_sim is None:
    raise RuntimeError("After normalization/mapping, no simulated samples produced.")

# Shuffle consistently
idx = rng.permutation(len(X_sim))
X_sim, y_sim = X_sim[idx], y_sim[idx]

print("SIM dataset:", X_sim.shape, y_sim.shape)
print("SIM label counts:", dict(zip(*np.unique(y_sim, return_counts=True))))

# =============================
# 3) Split SIM into train/test/final (70/15/15)
# =============================
(X_train_sim, y_train_sim), (X_test_sim, y_test_sim), (X_final_sim, y_final_sim) = stratified_split(X_sim, y_sim, SEED)
print("SIM splits -> Train:", X_train_sim.shape, "Test:", X_test_sim.shape, "Final:", X_final_sim.shape)

# =============================
# 4) Load EXISTING Router splits (already numeric) — DO NOT REMAP
# =============================
# Main router
rt_train_X_path = os.path.join(train_dir, "Router_training_X.npy")
rt_train_y_path = os.path.join(train_dir, "Router_training_y.npy")
rt_test_X_path  = os.path.join(test_dir,  "Router_test_X.npy")
rt_test_y_path  = os.path.join(test_dir,  "Router_test_y.npy")
rt_final_X_path = os.path.join(final_dir, "Router_final test_X.npy")
rt_final_y_path = os.path.join(final_dir, "Router_final test_y.npy")

X_train_old, y_train_old = load_router_split(rt_train_X_path, rt_train_y_path)
X_test_old,  y_test_old  = load_router_split(rt_test_X_path,  rt_test_y_path)
X_final_old, y_final_old = load_router_split(rt_final_X_path, rt_final_y_path)

# Router OCC (optional legacy)
rtocc_train_X_path = os.path.join(train_dir, "RouterOCC_training_X.npy")
rtocc_train_y_path = os.path.join(train_dir, "RouterOCC_training_y.npy")
rtocc_test_X_path  = os.path.join(test_dir,  "RouterOCC_test_X.npy")
rtocc_test_y_path  = os.path.join(test_dir,  "RouterOCC_test_y.npy")
rtocc_final_X_path = os.path.join(final_dir, "RouterOCC_final test_X.npy")
rtocc_final_y_path = os.path.join(final_dir, "RouterOCC_final test_y.npy")

X_train_occ, y_train_occ = load_router_split(rtocc_train_X_path, rtocc_train_y_path)
X_test_occ,  y_test_occ  = load_router_split(rtocc_test_X_path,  rtocc_test_y_path)
X_final_occ, y_final_occ = load_router_split(rtocc_final_X_path, rtocc_final_y_path)

# =============================
# 5) Combine (SIM + Router + RouterOCC) -> RouterV2.0
# =============================
def combine_three(simX, simy, rx, ry, roccx, roccy):
    X12, y12 = combine(simX, simy, rx, ry)
    X123, y123 = combine(X12, y12, roccx, roccy)
    return X123, y123

X_train_v2, y_train_v2 = combine_three(X_train_sim, y_train_sim, X_train_old, y_train_old, X_train_occ, y_train_occ)
X_test_v2,  y_test_v2  = combine_three(X_test_sim,  y_test_sim,  X_test_old,  y_test_old,  X_test_occ,  y_test_occ)
X_final_v2, y_final_v2 = combine_three(X_final_sim, y_final_sim, X_final_old, y_final_old, X_final_occ, y_final_occ)

print("RouterV2.0 -> Train:", X_train_v2.shape, "Test:", X_test_v2.shape, "Final:", X_final_v2.shape)
print("RouterV2.0 label counts (train):", dict(zip(*np.unique(y_train_v2, return_counts=True))))

# =============================
# 6) Save RouterV2.0 files
# =============================
np.save(os.path.join(train_dir, "RouterV2.0_training_X.npy"), X_train_v2)
np.save(os.path.join(train_dir, "RouterV2.0_training_y.npy"), y_train_v2)

np.save(os.path.join(test_dir,  "RouterV2.0_test_X.npy"), X_test_v2)
np.save(os.path.join(test_dir,  "RouterV2.0_test_y.npy"), y_test_v2)

np.save(os.path.join(final_dir, "RouterV2.0_final test_X.npy"), X_final_v2)
np.save(os.path.join(final_dir, "RouterV2.0_final test_y.npy"), y_final_v2)

print("✅ Saved RouterV2.0 datasets (training/test/final).")

# =============================
# 7) CSV for RouterV2.0 training (inspection)
# =============================
rows = []
T = X_train_v2.shape[1]  # should be 4
for seq in range(len(X_train_v2)):
    lab = int(y_train_v2[seq])  # 0/1/2/3
    for t in range(T):
        rows.append({
            "Sequence":    seq,
            "Time":        t,
            "Temperature": float(X_train_v2[seq, t, 0]),
            "Pressure":    float(X_train_v2[seq, t, 1]),
            "RPM":         float(X_train_v2[seq, t, 2]),
            "Vibration":   float(X_train_v2[seq, t, 3]),
            "RouterLabel": lab
        })

df = pd.DataFrame(rows, columns=["Sequence","Time","Temperature","Pressure","RPM","Vibration","RouterLabel"])
csv_path = os.path.join(train_dir, "RouterV2.0_train.csv")
df.to_csv(csv_path, index=False)
print(f"📝 RouterV2.0 Training CSV saved: {csv_path}  (sequences={len(X_train_v2)}, rows={len(rows)})")


SIM dataset: (38570, 4, 4) (38570,)
SIM label counts: {0: 665, 1: 4655, 2: 13965, 3: 19285}
SIM splits -> Train: (26999, 4, 4) Test: (5785, 4, 4) Final: (5786, 4, 4)
RouterV2.0 -> Train: (236999, 4, 4) Test: (44785, 4, 4) Final: (44786, 4, 4)
RouterV2.0 label counts (train): {0: 8465, 1: 11259, 2: 63775, 3: 153500}
✅ Saved RouterV2.0 datasets (training/test/final).
📝 RouterV2.0 Training CSV saved: ../../../data/training/RouterV2.0_train.csv  (sequences=236999, rows=947996)
